# Hands-On Lab: Agentic Memory Buffers in Google ADK

*Short-term memory, session memory, and long-term Memory Bank*

You will build and run an AI agent with the Google Agent Development Kit (`google-adk`), store its memory in Vertex AI Agent Engine, and authenticate with a GCP or Gemini API key.

### Goal

* Build an ADK agent whose tools write to all three memory buffers.
* Inspect memory on every turn with the in-notebook **Memory X-Ray** panel.
* Watch the same data appear in the Cloud Console **Sessions** and **Memories** tabs.

**Time:** 30 to 40 minutes. Modules 2 and 3 follow one incident ticket, INC-4092.

## The Three Memory Buffers at a Glance

An **ADK Primitive** is a building block that ships with `google-adk`, so you don't write your own storage code. This lab uses three kinds:

* **State key prefixes** (`temp:`, no prefix, `user:`) set how long a value lives.
* **Context window controls** (`ContextFilterPlugin`, `EventsCompactionConfig`) set how much history the model sees.
* **Managed services** (`SessionService`, `MemoryService`) set where data is stored and how it is recalled.

| Memory Buffer | Lifetime & Purpose | Cloud Console View | ADK Primitive |
| :--- | :--- | :--- | :--- |
| **1. Short-Term**<br>*Working buffer* | One turn. Holds scratch values and limits history so prompts stay small. | Not saved. Visible only in the notebook X-Ray. | `temp:*` keys: scratchpad, deleted after the turn.<br><br>`ContextFilterPlugin`: keeps the last N turns.<br><br>`EventsCompactionConfig`: summarizes older turns. |
| **2. Session**<br>*Episodic buffer* | One conversation. Tracks the current task and chat history, survives restarts, and stays separate from other sessions. | **Sessions** tab: one row per session. Click a row for the **Event Log** and **State**. | Unprefixed keys: saved to this session only.<br><br>`user:*` keys: shared by all of this user's sessions.<br><br>`Session.events`: the message log.<br><br>`SessionService`: saves and reloads sessions. |
| **3. Long-Term**<br>*Memory Bank* | Across sessions. Stores facts from past sessions and recalls them later. On Vertex AI, newer facts replace older ones. | **Memories** tab: one row per fact. **View revisions** shows its history. | `add_events_to_memory()`: saves a session's facts.<br><br>`preload_memory_tool`: recalls relevant facts each turn.<br><br>`load_memory_tool`: lets the model search on demand. |

---

## Step 1: Install Dependencies

Installs `google-adk` and the Vertex AI SDK.

In [ ]:
# @title Step 1: Install Google ADK and the Vertex AI SDK
# ipywidgets is not upgraded: Colab ships a compatible version, and upgrading it breaks widgets.
!pip install -q "google-adk" "google-cloud-aiplatform[agent_engines]>=1.148.1"

import google.adk
print(f"google-adk {google.adk.__version__} installed.")

---

## Step 2A: Configure Your API Key

1. Choose `API_KEY_MODE`:
   * **Vertex AI Express Mode key:** use this to see the Cloud Console views.
   * **Gemini API key (AI Studio):** works only with local mode in Step 2B.
2. Provide the key in one of three ways:
   * Colab **Secrets** (key icon in the left sidebar), named `GOOGLE_API_KEY`.
   * The `API_KEY_OVERRIDE` field.
   * The prompt that appears when the cell runs.

In [ ]:
# @title Step 2A: Configure Your GCP / Gemini API Key
import getpass
import os

API_KEY_MODE = "GCP Vertex AI Express Mode API Key"  # @param ["GCP Vertex AI Express Mode API Key", "Gemini API Key (AI Studio)"]
MODEL_NAME = "gemini-3.8-flash"  # @param {type:"string"}
API_KEY_OVERRIDE = ""  # @param {type:"string"}

APP_NAME = "memory_lab_app"
USER_ID = "dev_alex"

try:
    from google.colab import userdata  # type: ignore
    _secret_key = (userdata.get("GOOGLE_API_KEY") or "").strip()
except Exception:
    _secret_key = ""

ACTIVE_API_KEY = (
    API_KEY_OVERRIDE.strip()
    or _secret_key
    or os.environ.get("GOOGLE_API_KEY", "").strip()
    or getpass.getpass("Enter your GCP / Gemini API key (GOOGLE_API_KEY): ").strip()
)
USING_EXPRESS_KEY = "Vertex AI" in API_KEY_MODE
os.environ["GOOGLE_API_KEY"] = ACTIVE_API_KEY
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "1" if USING_EXPRESS_KEY else "0"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = os.environ["GOOGLE_GENAI_USE_ENTERPRISE"]
print(f"API key configured ({API_KEY_MODE}) | Model: {MODEL_NAME} | User: {USER_ID}")

---

## Step 2B: Connect Session and Memory Storage

Pick a `STORAGE_MODE`:

| Mode | Sign-in | Cloud Console views | Use when |
| :--- | :--- | :--- | :--- |
| Vertex AI Express Mode | Express Mode key from Step 2A | Yes | Default. No GCP project needed. |
| Vertex AI with GCP project | Your Google account | Yes | You have a project with Vertex AI enabled. |
| Local in-memory | None | No | Offline practice. Data is lost on restart. |

What the cell does:

* If `AGENT_ENGINE_ID` is blank, it creates a small Agent Engine (Sessions and Memory Bank only, no agent code).
* If you run it again, it reuses that engine.
* If cloud setup fails, it stops with an error. It never switches to local mode on its own.
* It prints buttons that open your engine's **Sessions** and **Memories** tabs.

> **Tip:** keep the Console open in a second tab and click **Refresh** after each lab cell. The Console menu may say **Vertex AI → Agent Engine** or **Agent Platform → Runtimes**, so the cell prints links for both.

In [ ]:
# @title Step 2B: Connect SessionService and MemoryService (+ Cloud Console links) { display-mode: "form" }
import json
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from IPython.display import HTML, display

STORAGE_MODE = "Vertex AI Express Mode (API key)"  # @param ["Vertex AI Express Mode (API key)", "Vertex AI with GCP project (Colab sign-in)", "Local in-memory (no Cloud Console)"]
AGENT_ENGINE_ID = ""  # @param {type:"string"}
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCP_LOCATION = "us-central1"  # @param {type:"string"}

USING_CLOUD_AGENT_ENGINE = not STORAGE_MODE.startswith("Local")
USE_PROJECT_AUTH = STORAGE_MODE.startswith("Vertex AI with GCP project")
ACTIVE_PROJECT = GCP_PROJECT_ID.strip()
ACTIVE_LOCATION = GCP_LOCATION.strip() or "us-central1"
ACTIVE_ENGINE_ID = AGENT_ENGINE_ID.strip()


def vertex_client():
    """Returns a Vertex AI client for the selected auth path."""
    import vertexai

    if USE_PROJECT_AUTH:
        return vertexai.Client(project=ACTIVE_PROJECT, location=ACTIVE_LOCATION)
    return vertexai.Client(api_key=ACTIVE_API_KEY)


def _service_kwargs() -> dict:
    if USE_PROJECT_AUTH:
        return dict(project=ACTIVE_PROJECT, location=ACTIVE_LOCATION, agent_engine_id=ACTIVE_ENGINE_ID)
    return dict(agent_engine_id=ACTIVE_ENGINE_ID, express_mode_api_key=ACTIVE_API_KEY)


def make_session_service():
    """Builds a brand-new SessionService object (used again in Module 2)."""
    if USING_CLOUD_AGENT_ENGINE:
        return VertexAiSessionService(**_service_kwargs())
    return InMemorySessionService()


def make_memory_service():
    if USING_CLOUD_AGENT_ENGINE:
        return VertexAiMemoryBankService(**_service_kwargs())
    return InMemoryMemoryService()


if USING_CLOUD_AGENT_ENGINE:
    try:
        if USE_PROJECT_AUTH:
            if not ACTIVE_PROJECT:
                raise ValueError("Set GCP_PROJECT_ID for the GCP project mode.")
            try:
                from google.colab import auth  # type: ignore
                auth.authenticate_user()
            except ImportError:
                pass  # Not in Colab: rely on Application Default Credentials.
            # With an API key in the environment, ADK keeps using Express Mode.
            os.environ.pop("GOOGLE_API_KEY", None)
            os.environ["GOOGLE_CLOUD_PROJECT"] = ACTIVE_PROJECT
            os.environ["GOOGLE_CLOUD_LOCATION"] = ACTIVE_LOCATION
        elif not USING_EXPRESS_KEY:
            raise ValueError(
                "Express Mode storage needs a Vertex AI Express Mode API key. "
                "Change API_KEY_MODE in Step 2A, or pick another STORAGE_MODE."
            )
        os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "1"
        os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"

        cache_key = f"{STORAGE_MODE}|{ACTIVE_PROJECT}|{ACTIVE_LOCATION}"
        cached = json.loads(os.environ.get("MEMORY_LAB_ENGINE", "{}"))
        if not ACTIVE_ENGINE_ID and cached.get("key") == cache_key:
            ACTIVE_PROJECT, ACTIVE_LOCATION, ACTIVE_ENGINE_ID = cached["project"], cached["location"], cached["id"]
            print(f"Reusing the Agent Engine created earlier in this runtime: {ACTIVE_ENGINE_ID}")
        elif not ACTIVE_ENGINE_ID:
            print("Creating a lightweight Agent Engine (Sessions + Memory Bank). This takes about a minute...")
            created = vertex_client().agent_engines.create(config={"display_name": "memory-lab"})
            parts = created.api_resource.name.split("/")  # projects/{p}/locations/{l}/reasoningEngines/{id}
            ACTIVE_PROJECT, ACTIVE_LOCATION, ACTIVE_ENGINE_ID = parts[1], parts[3], parts[5]
            os.environ["MEMORY_LAB_ENGINE"] = json.dumps(
                {"key": cache_key, "project": ACTIVE_PROJECT, "location": ACTIVE_LOCATION, "id": ACTIVE_ENGINE_ID}
            )
        session_service = make_session_service()
        memory_service = make_memory_service()
        print(f"Connected to Vertex AI Agent Engine {ACTIVE_ENGINE_ID} ({ACTIVE_LOCATION}).")
    except Exception as exc:
        raise RuntimeError(
            f"Cloud setup failed: {exc!r}\n"
            "Fix the error above, or set STORAGE_MODE to 'Local in-memory' to continue without Cloud Console."
        ) from exc
else:
    session_service = make_session_service()
    memory_service = make_memory_service()
    print("Using local InMemorySessionService and InMemoryMemoryService. Cloud Console views are not available.")


def get_cloud_console_urls() -> dict[str, str]:
    """Returns Cloud Console URLs for this engine's Sessions and Memories tabs (new and legacy paths)."""
    q = f"?project={ACTIVE_PROJECT}" if ACTIVE_PROJECT else ""
    tail = f"locations/{ACTIVE_LOCATION}/agent-engines/{ACTIVE_ENGINE_ID}"
    new = f"https://console.cloud.google.com/agent-platform/runtimes/{tail}"
    old = f"https://console.cloud.google.com/vertex-ai/agents/agent-engines/{tail}"
    return {
        "sessions_tab": f"{new}/sessions{q}",
        "memories_tab": f"{new}/memories{q}",
        "sessions_legacy": f"{old}/sessions{q}",
        "memories_legacy": f"{old}/memories{q}",
    }


def render_cloud_console_banner() -> None:
    if not USING_CLOUD_AGENT_ENGINE:
        display(HTML(
            "<div style=\"font-family:'Google Sans',Roboto,sans-serif;border:2px solid #f59e0b;border-radius:10px;"
            "background:#fffbeb;padding:12px 16px;margin:8px 0;font-size:13px;color:#78350f;\">"
            "<b>LOCAL IN-MEMORY MODE.</b> The labs still run, but nothing is written to Google Cloud, so the "
            "Cloud Console steps will show nothing. Use Express Mode or a GCP project in Step 2B to see them.</div>"
        ))
        return
    urls = get_cloud_console_urls()
    btn = "text-decoration:none;color:white;padding:7px 14px;border-radius:6px;font-size:12.5px;font-weight:600;"
    alt = "text-decoration:none;background:#e2e8f0;color:#1e293b;padding:7px 12px;border-radius:6px;font-size:12px;"
    display(HTML(f"""
    <div style="font-family:'Google Sans',Roboto,sans-serif;border:1px solid #cbd5e1;border-radius:10px;background:#f8fafc;padding:14px 18px;margin:8px 0;">
      <div style="font-weight:700;font-size:15px;color:#0f172a;margin-bottom:6px;">Google Cloud Console: Agent Engine {ACTIVE_ENGINE_ID}</div>
      <div style="font-size:13px;color:#334155;margin-bottom:10px;">Open these in a second tab. After each lab cell, click <b>Refresh</b> in the Console table.</div>
      <div style="display:flex;gap:10px;flex-wrap:wrap;">
        <a href="{urls['sessions_tab']}" target="_blank" style="{btn}background:#0284c7;">Open Sessions tab</a>
        <a href="{urls['memories_tab']}" target="_blank" style="{btn}background:#7c3aed;">Open Memories tab</a>
        <a href="{urls['sessions_legacy']}" target="_blank" style="{alt}">Sessions (legacy Vertex AI path)</a>
        <a href="{urls['memories_legacy']}" target="_blank" style="{alt}">Memories (legacy Vertex AI path)</a>
      </div>
    </div>"""))


render_cloud_console_banner()

---

## Step 3: Tools, Helpers, Chat Loop, and Agent

Tools save data with `tool_context.state[key] = value`. The prefix on the key decides where the value lives:

| Prefix | Example | Where it lives |
| :--- | :--- | :--- |
| `temp:` | `temp:daily_cost` | This turn only. Deleted before saving. |
| *(none)* | `current_ticket` | This session, in `SessionService`. |
| `user:` | `user:preferred_language` | Every session of this user, in `SessionService`. |

`user:` keys are not Memory Bank. Memory Bank only receives facts when you save a session in Module 3.

Step 3C shows the chat loop: one `runner.run_async()` call per turn.

In [ ]:
# @title Step 3A: Define One ADK Tool per Memory Buffer
from google.adk.tools.tool_context import ToolContext


def stage_scratchpad_note(key: str, note: str, tool_context: ToolContext) -> dict:
    """Buffer 1 (Short-Term): 'temp:' keys live ONLY for the current turn."""
    tool_context.state[f"temp:{key.removeprefix('temp:')}"] = note
    return {"short_term_staged": {f"temp:{key.removeprefix('temp:')}": note}}


def update_session_task(ticket_id: str, step: str, tool_context: ToolContext) -> dict:
    """Buffer 2 (Session): Unprefixed keys persist across turns in ONE session."""
    tool_context.state["current_ticket"] = ticket_id
    tool_context.state["current_step"] = step
    return {"session_saved": {"current_ticket": ticket_id, "current_step": step}}


def save_user_preference(key: str, value: str, tool_context: ToolContext) -> dict:
    """User-scoped state: 'user:' keys are shared by ALL sessions of this user."""
    tool_context.state[f"user:{key.removeprefix('user:')}"] = value
    return {"user_state_saved": {f"user:{key.removeprefix('user:')}": value}}


print("Defined 3 ADK tools: stage_scratchpad_note, update_session_task, save_user_preference")

In [ ]:
# @title Step 3B: Load the Memory X-Ray and Lab Helpers { display-mode: "form" }
import html
import time
from typing import Any, Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse

XRAY_HISTORY: list[dict[str, Any]] = []
SESSION_LABELS: dict[str, str] = {}


def _extract_preloaded_memories(llm_request: LlmRequest) -> list[str]:
    """Finds the <PAST_CONVERSATIONS> block that preload_memory_tool adds to the request."""
    texts = []
    if llm_request.config and llm_request.config.system_instruction:
        texts.append(str(llm_request.config.system_instruction))
    for content in llm_request.contents or []:
        texts.extend(p.text for p in content.parts or [] if getattr(p, "text", None))
    memories = []
    for text in texts:
        if "<PAST_CONVERSATIONS>" in text:
            block = text.split("<PAST_CONVERSATIONS>", 1)[1].split("</PAST_CONVERSATIONS>", 1)[0]
            memories.extend(line.strip() for line in block.splitlines() if line.strip())
    return memories


def memory_xray_callback(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Records a snapshot of all 3 memory buffers right before each Gemini call."""
    session = getattr(callback_context, "session", None) or callback_context._invocation_context.session
    state = callback_context.state.to_dict()
    window_msgs = []
    for c in llm_request.contents or []:
        parts = []
        for p in c.parts or []:
            if getattr(p, "text", None):
                t = p.text.strip().replace("\n", " ")
                parts.append(t[:90] + ("..." if len(t) > 90 else ""))
            elif getattr(p, "function_call", None):
                parts.append(f"call:{p.function_call.name}()")
            elif getattr(p, "function_response", None):
                parts.append(f"result:{p.function_response.name}")
        window_msgs.append(f"<b>{html.escape(str(c.role))}:</b> {html.escape(' | '.join(parts))}")
    XRAY_HISTORY.append({
        "session_id": session.id,
        "session_label": SESSION_LABELS.get(session.id, session.id),
        "temp_state": {k: v for k, v in state.items() if k.startswith("temp:")},
        "session_state": {k: v for k, v in state.items() if not k.startswith(("temp:", "user:", "app:"))},
        "user_state": {k: v for k, v in state.items() if k.startswith("user:")},
        "window_count": len(llm_request.contents or []),
        "window_msgs": window_msgs,
        "preloaded_memories": _extract_preloaded_memories(llm_request),
        "total_session_events": len(session.events),
    })
    return None


def _fmt_kv(d: dict[str, Any], empty_msg: str) -> str:
    if not d:
        return f"<i style='color:#64748b;'>{empty_msg}</i>"
    return "".join(
        f"<div style='margin:2px 0;'><code style='background:#f1f5f9;padding:1px 5px;border-radius:4px;'>"
        f"{html.escape(str(k))}</code>: <b>{html.escape(json.dumps(v))}</b></div>"
        for k, v in d.items()
    )


def render_memory_xray(snapshot: Optional[dict[str, Any]] = None) -> None:
    """Renders the 3-column Memory X-Ray card for the latest (or given) snapshot."""
    snap = snapshot or (XRAY_HISTORY[-1] if XRAY_HISTORY else None)
    if not snap:
        return
    links = ""
    if USING_CLOUD_AGENT_ENGINE:
        urls = get_cloud_console_urls()
        links = (
            f"<a href='{urls['sessions_tab']}' target='_blank' style='font-size:11.5px;color:#0284c7;font-weight:600;'>Console: Sessions</a>"
            f" &nbsp; <a href='{urls['memories_tab']}' target='_blank' style='font-size:11.5px;color:#7c3aed;font-weight:600;'>Console: Memories</a>"
        )
    win_preview = "<br>".join(snap["window_msgs"][-4:]) or "<i>Empty</i>"
    mem_preview = (
        "<br>".join(f"- {html.escape(m[:100])}" for m in snap["preloaded_memories"][-3:])
        or "<i style='color:#64748b;'>Nothing recalled this turn</i>"
    )
    box = "border-radius:8px;padding:9px;"
    display(HTML(f"""
    <div style="font-family:'Google Sans',Roboto,sans-serif;border:1px solid #cbd5e1;border-radius:10px;background:#f8fafc;padding:12px;margin:8px 0;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;flex-wrap:wrap;gap:6px;">
        <div><b style="color:#1e293b;font-size:13.5px;">Memory X-Ray</b>
          <span style="font-size:12px;color:#475569;margin-left:8px;">Session <code>{html.escape(str(snap['session_label']))}</code>
          (ID <code>{html.escape(str(snap['session_id']))}</code>)</span></div>
        <div>{links}</div>
      </div>
      <div style="display:grid;grid-template-columns:repeat(3, 1fr);gap:10px;font-size:12px;">
        <div style="background:#fffbeb;border:1px solid #fde68a;{box}">
          <div style="font-weight:700;color:#b45309;margin-bottom:4px;">1. Short-Term (Working Buffer)</div>
          <div><b><code>temp:*</code> keys this turn:</b></div>{_fmt_kv(snap['temp_state'], 'None')}
          <hr style="border:none;border-top:1px dashed #fcd34d;margin:6px 0;">
          <div><b>Sent to Gemini:</b> {snap['window_count']} messages (session has {snap['total_session_events']} events)</div>
          <div style="margin-top:3px;font-size:11px;color:#475569;max-height:75px;overflow-y:auto;">{win_preview}</div>
        </div>
        <div style="background:#f0f9ff;border:1px solid #bae6fd;{box}">
          <div style="font-weight:700;color:#0369a1;margin-bottom:4px;">2. Session Memory (SessionService)</div>
          <div><b>Session state:</b></div>{_fmt_kv(snap['session_state'], 'No session keys yet')}
          <hr style="border:none;border-top:1px dashed #7dd3fc;margin:6px 0;">
          <div><b>User state (<code>user:*</code>):</b></div>{_fmt_kv(snap['user_state'], 'No user:* keys yet')}
        </div>
        <div style="background:#f5f3ff;border:1px solid #ddd6fe;{box}">
          <div style="font-weight:700;color:#6d28d9;margin-bottom:4px;">3. Long-Term Memory (Memory Bank)</div>
          <div><b>Recalled by <code>preload_memory_tool</code>:</b></div>
          <div style="margin-top:3px;font-size:11px;color:#475569;max-height:90px;overflow-y:auto;">{mem_preview}</div>
        </div>
      </div>
    </div>"""))


async def create_lab_session(label: str):
    """Creates a new session with a readable, unique ID such as session_alpha-1700000000."""
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=f"{label}-{int(time.time())}"
        )
    except Exception as exc:  # Some backends only accept server-assigned IDs.
        print(f"(Custom session ID rejected: {exc!r}. Using a server-assigned ID.)")
        session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    SESSION_LABELS[session.id] = label
    print(f"Created session `{label}` (Session ID: {session.id})")
    return session


async def get_lab_session(session):
    """Reloads the latest copy of a session from SessionService."""
    reloaded = await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=session.id)
    SESSION_LABELS.setdefault(reloaded.id, SESSION_LABELS.get(session.id, session.id))
    return reloaded


async def save_session_to_memory_bank(session) -> None:
    """Extracts facts from a session into Memory Bank and waits until they are stored."""
    latest = await get_lab_session(session)
    label = SESSION_LABELS.get(latest.id, latest.id)
    print(f"Saving `{label}` ({len(latest.events)} events) to Memory Bank. This can take up to a minute...")
    try:
        await memory_service.add_events_to_memory(
            app_name=latest.app_name,
            user_id=latest.user_id,
            events=latest.events,
            session_id=latest.id,
            custom_metadata={"wait_for_completion": True},  # Ignored by the in-memory service.
        )
    except (AttributeError, NotImplementedError):  # Older google-adk releases.
        await memory_service.add_session_to_memory(latest)
    where = "Memory Bank"
    if USING_CLOUD_AGENT_ENGINE:
        where = (
            f"<a href='{get_cloud_console_urls()['memories_tab']}' target='_blank' style='color:#6d28d9;font-weight:700;'>"
            f"Console Memories tab</a> (click Refresh, filter <code>user_id:{USER_ID}</code>)"
        )
    display(HTML(
        f"<div style=\"font-family:'Google Sans',Roboto,sans-serif;background:#f5f3ff;border-left:4px solid #7c3aed;"
        f"padding:8px 12px;margin:6px 0;font-size:12.5px;\"><b>Saved <code>{html.escape(label)}</code> to long-term memory.</b>"
        f" Check the {where}.</div>"
    ))


print("Loaded helpers: Memory X-Ray, create_lab_session(), get_lab_session(), save_session_to_memory_bank()")

In [ ]:
# @title Step 3C: The Chat Loop (how one turn runs through ADK)
from google.genai import types


async def chat(runner, session, prompt: str, *, show_xray: bool = True) -> str:
    """Sends one user message through the Runner and prints tool calls and the reply."""
    label = SESSION_LABELS.get(session.id, session.id)
    print(f"\n[{label}] User: {prompt}")
    message = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    reply = ""
    # run_async loads the session, calls Gemini and tools, and saves every event back to SessionService.
    async for event in runner.run_async(user_id=USER_ID, session_id=session.id, new_message=message):
        for call in event.get_function_calls():
            print(f"  tool call -> {call.name}({dict(call.args or {})})")
        if event.is_final_response() and event.content and event.content.parts:
            reply = "".join(p.text or "" for p in event.content.parts)
    print(f"[{label}] Agent: {reply}")
    if show_xray:
        render_memory_xray()
    return reply

In [ ]:
# @title Step 3D: Create the ADK LlmAgent and Runner
from google.adk.agents import LlmAgent
from google.adk.apps.app import App
from google.adk.runners import Runner
from google.adk.tools.load_memory_tool import load_memory_tool
from google.adk.tools.preload_memory_tool import preload_memory_tool

AGENT_INSTRUCTION = """You are a concise Cloud Architect Assistant running on Google ADK.
- Call `stage_scratchpad_note` for any math or cost calculation (stores in `temp:*`).
- Call `update_session_task` when starting or updating a ticket (stores in session state).
- Call `save_user_preference` when the user states or updates a preference like language (stores in `user:*`).

Live state injected by the ADK runtime:
- Active Ticket: {current_ticket?} (Step: {current_step?})
- Preferred Language: {user:preferred_language?}"""


def create_runner(plugins=None, compaction_config=None) -> Runner:
    agent = LlmAgent(
        name="cloud_architect_assistant",
        model=MODEL_NAME,
        instruction=AGENT_INSTRUCTION,
        tools=[stage_scratchpad_note, update_session_task, save_user_preference, preload_memory_tool, load_memory_tool],
        before_model_callback=memory_xray_callback,
    )
    app = App(name=APP_NAME, root_agent=agent, plugins=plugins or [], events_compaction_config=compaction_config)
    return Runner(app=app, session_service=session_service, memory_service=memory_service)


runner = create_runner()
print("ADK Runner ready.")

---

## Module 1: Short-Term Memory

Short-term memory keeps each prompt small and focused.

* **Lab 1A:** a tool saves a `temp:` value. It works for the rest of the turn, then ADK deletes it before saving.
* **Lab 1B:** three ways to limit how much history the model sees.

> **Predict first:** after Lab 1A, will `temp:daily_cost` be in the saved session state?

#### Check it in Cloud Console

1. Open the **Sessions** tab and click **Refresh**.
2. Open the `stm_temp_demo` session.
3. The **Event Log** shows the tool call, but **State** has no `temp:` key.

In [ ]:
# @title Lab 1A: Stage temp:* Data During a Turn, Then Confirm It Was Not Saved
PROMPT = "Calculate the daily token cost for 50,000 requests at 4,000 tokens each ($0.15 per 1M tokens), and stage your math in the scratchpad."  # @param {type:"string"}

stm_session = await create_lab_session("stm_temp_demo")
await chat(runner, stm_session, PROMPT)

seen = [s["temp_state"] for s in XRAY_HISTORY if s["session_id"] == stm_session.id and s["temp_state"]]
saved_state = (await get_lab_session(stm_session)).state
print("temp:* keys the model could see during the turn:", seen[-1] if seen else "(none captured by the X-Ray)")
print("State saved in SessionService:", saved_state)
assert not any(k.startswith("temp:") for k in saved_state), "temp:* keys should never be saved"
print("Confirmed: temp:* was not saved to SessionService.")

### Lab 1B: Context Window Control

Each turn adds events to the session. Sending all of them to Gemini every turn makes prompts slower and costlier, and old turns can distract the model. ADK has three ways to control what the model sees:

| Option | ADK Primitive | What the model sees | Trade-off |
| :--- | :--- | :--- | :--- |
| 1. Unbounded | None (default) | Every past turn | Full context, but cost grows each turn |
| 2. Sliding window | `ContextFilterPlugin` | The last N turns | Cheap, but older facts are dropped |
| 3. LLM compaction | `EventsCompactionConfig` | A summary of older turns, plus recent turns | Keeps old facts, but costs an extra Gemini call |

Each option runs the same 4-turn incident in its own session. The last cell compares them.

> **Predict first:** the key clue (Cloud NAT ports at 100%) arrives in Turn 3. With a sliding window of N = 1, can the agent still find the root cause in Turn 4? Will it remember the Turn 1 symptom?

#### Check it in Cloud Console

1. Open the `window_unbounded`, `window_sliding`, and `window_compaction` sessions.
2. All three store every turn in the **Event Log**. Window control changes what is sent to the model, not what is saved.
3. The compaction session also has one extra summary event.

In [ ]:
# @title Lab 1B Setup: Shared 4-Turn Incident and Window Tracker
from google.adk.apps.app import EventsCompactionConfig
from google.adk.apps.llm_event_summarizer import LlmEventSummarizer
from google.adk.models import Gemini
from google.adk.plugins.context_filter_plugin import ContextFilterPlugin

INCIDENT_TURNS = [
    "Turn 1: Our API gateway in us-east1 is seeing 502 Bad Gateway spikes.",
    "Turn 2: GKE backend pod CPU utilization is only at 18%.",
    "Turn 3: Cloud NAT ephemeral port usage hit 100% saturation.",
    "Turn 4: Based on everything so far, what is the root cause and fix? Mention the original symptom.",
]
WINDOW_RESULTS = {}


async def run_window_demo(mode: str, window_runner, label: str) -> None:
    """Runs the 4 incident turns and records how many messages Gemini saw per turn."""
    demo_session = await create_lab_session(label)
    sizes, answer = [], ""
    for idx, turn_text in enumerate(INCIDENT_TURNS, start=1):
        first_snapshot = len(XRAY_HISTORY)
        answer = await chat(window_runner, demo_session, turn_text, show_xray=(idx == len(INCIDENT_TURNS)))
        sizes.append(XRAY_HISTORY[first_snapshot]["window_count"] if len(XRAY_HISTORY) > first_snapshot else None)
    WINDOW_RESULTS[mode] = {"window_sizes": sizes, "answer": answer}
    print(f"\n[{mode}] Messages sent to Gemini per turn: {sizes}")

In [ ]:
# @title Option 1: Unbounded (Full History, the Default)
# No plugin, no compaction: Gemini receives every past turn on every call.
unbounded_runner = create_runner()

await run_window_demo("1. Unbounded", unbounded_runner, "window_unbounded")

In [ ]:
# @title Option 2: Sliding Window (ContextFilterPlugin)
KEEP_LAST_N_TURNS = 1  # @param {type:"slider", min:1, max:4, step:1}

# Keeps only the last N turns in the prompt. Tool call/response pairs are never split.
sliding_runner = create_runner(
    plugins=[ContextFilterPlugin(num_invocations_to_keep=KEEP_LAST_N_TURNS, remove_amount=1)]
)

await run_window_demo(f"2. Sliding Window (N={KEEP_LAST_N_TURNS})", sliding_runner, "window_sliding")

In [ ]:
# @title Option 3: LLM Event Compaction (EventsCompactionConfig)
# Every 2 turns, Gemini summarizes older events into one compact summary event.
# overlap_size=1 keeps the most recent turn in full alongside the summary.
compaction_runner = create_runner(
    compaction_config=EventsCompactionConfig(
        compaction_interval=2,
        overlap_size=1,
        summarizer=LlmEventSummarizer(llm=Gemini(model=MODEL_NAME)),
    )
)

await run_window_demo("3. LLM Compaction", compaction_runner, "window_compaction")

In [ ]:
# @title Compare All 3 Context Window Options
import pandas as pd

pd.set_option("display.max_colwidth", 220)
comparison = pd.DataFrame([
    {"Option": mode, **{f"Turn {i} msgs": n for i, n in enumerate(r["window_sizes"], 1)}, "Turn 4 answer": r["answer"]}
    for mode, r in WINDOW_RESULTS.items()
])
display(comparison)
print("Fewer messages = cheaper, faster prompts. Check whether each answer still finds the root cause (Cloud NAT) AND the original symptom (502s).")

---

## Module 2: Session Memory

Scenario: you are on call for ticket INC-4092, a Cloud Run cold-start spike. Session state (`current_ticket`, `current_step`) and the event log are saved by `SessionService`, not by the `Runner`.

* **Lab 2A:** open INC-4092, restart the `Runner`, and continue the same ticket. Then check whether a brand-new `SessionService` can still load the session.
* **Lab 2B:** open a second session for the same user and see what carries over.

> **Predict first:** will the new `SessionService` find the session in cloud mode? In local mode? Will `session_beta` see `current_ticket`? Will it see `user:preferred_language`?

#### Check it in Cloud Console

Open the **Sessions** tab, filter by `userId:dev_alex`, and click **Refresh**.

1. After Lab 2A: open `session_alpha`. **State** shows `current_ticket: "INC-4092"`.
2. After the restart turn: **Last active** moves forward.
3. After Lab 2B: a separate `session_beta` row appears.

> **Note:** whether `user:` keys carry over to a new session depends on the backend. Local mode copies them. On Vertex AI, check the **State** of `session_beta`.

In [ ]:
# @title Lab 2A: Open INC-4092, Restart the Runner, and Rehydrate the Session
TICKET_ID = "INC-4092"  # @param {type:"string"}
INITIAL_LANGUAGE = "Python"  # @param ["Python", "Go", "Rust", "TypeScript"]

session_alpha = await create_lab_session("session_alpha")
await chat(runner, session_alpha, f"Open ticket {TICKET_ID} (Step 1: Tracing) for a Cloud Run cold-start spike, and save my preferred_language as {INITIAL_LANGUAGE}.")

# Simulate a server restart: discard the Runner and build a new one on the same SessionService.
del runner
runner = create_runner()
await chat(runner, session_alpha, "We just reconnected after a restart. What ticket and step are we on? Give me a 1-line log line in my preferred language.")

# Where does the session really live? Ask a brand-new SessionService object.
fresh_service = make_session_service()
found = await fresh_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_alpha.id)
print("A brand-new SessionService can load session_alpha:", found is not None)
print("(Cloud: True, because the session is stored in Agent Engine. Local: False, because it only existed in this Python process.)")

In [ ]:
# @title Lab 2B: Check Session Isolation in session_beta
session_beta = await create_lab_session("session_beta")
await chat(runner, session_beta, "Do I have an active ticket in this session? What is my preferred language?")

beta_state = (await get_lab_session(session_beta)).state
print("current_ticket in session_beta:", beta_state.get("current_ticket"))
print("user:preferred_language in session_beta:", beta_state.get("user:preferred_language"))

---

## Module 3: Long-Term Memory (Memory Bank)

In Lab 2B, `session_beta` knew nothing about INC-4092, because each session keeps its own events. To share facts with future sessions, you save a finished session to Memory Bank. Gemini extracts the facts, and `preload_memory_tool` recalls the relevant ones in later sessions.

* **Lab 3A:** save `session_alpha`, then ask about INC-4092 from a new session.
* **Lab 3B:** change your preferred language to Go, save again, and check what a new session knows.

> **Predict first:** in Lab 3B, which source gives the agent your current language: `user:` state or Memory Bank? Which source gives the INC-4092 history?

#### Check it in Cloud Console

Open the **Memories** tab, enter `user_id:dev_alex` in the scope filter, and click **Refresh**.

1. After Lab 3A: facts from `session_alpha` appear in the **Fact** column.
2. After Lab 3B: the language fact says Go, with a newer **Updated** time.
3. Open that row's menu and choose **View revisions** to see the older Python version.

> **Note:** Vertex AI Memory Bank merges conflicting facts, so it recalls only the current one. Local mode does a keyword search over raw messages, so it may recall both the old and the new language.

In [ ]:
# @title Lab 3A: Save session_alpha to Memory Bank, Then Recall It in session_gamma
CROSS_SESSION_QUESTION = "We debugged a Cloud Run cold-start incident in an earlier session. What ticket ID was that and what step did we reach?"  # @param {type:"string"}

await save_session_to_memory_bank(session_alpha)

session_gamma = await create_lab_session("session_gamma")
await chat(runner, session_gamma, CROSS_SESSION_QUESTION)

In [ ]:
# @title Lab 3B: Update a Preference, Save Again, and Verify in session_delta
UPDATED_LANGUAGE = "Go"  # @param ["Go", "Rust", "TypeScript", "Java"]

await chat(runner, session_gamma, f"Our team migrated away from our old language. Update my preferred_language to {UPDATED_LANGUAGE} for all new services.")
await save_session_to_memory_bank(session_gamma)

session_delta = await create_lab_session("session_delta")
await chat(runner, session_delta, "What is my preferred language now, and which language did I use on ticket INC-4092?")

---

## Module 4: Memory Workbench

Free play. The main controls:

* **Send Turn:** sends your prompt. The **Window (N)** slider sets the sliding window size.
* **New Session:** starts a fresh session, to test what carries over.
* **Restart Runner:** simulates a server restart.
* **Save to Memory Bank:** saves the session. Then refresh the **Memories** tab.

Errors appear in the output box below the buttons.

In [ ]:
# @title Launch the Memory Workbench { display-mode: "form" }
import asyncio
import ipywidgets as widgets

try:
    from google.colab import output as _colab_output  # type: ignore
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass


def _window_runner(n: int):
    return create_runner(plugins=[ContextFilterPlugin(num_invocations_to_keep=n, remove_amount=1)])


WORKBENCH = {"counter": 1, "window": 3, "session": await create_lab_session("playground_1")}
WORKBENCH["runner"] = _window_runner(WORKBENCH["window"])
render_cloud_console_banner()

prompt_box = widgets.Text(
    value="Calculate RAM for 8 pods at 4GB each (stage in scratchpad), and set ticket INC-9000 (Step 1: Capacity).",
    description="Prompt:",
    layout=widgets.Layout(width="98%"),
)
win_slider = widgets.IntSlider(value=WORKBENCH["window"], min=1, max=6, description="Window (N):")
btn_send = widgets.Button(description="Send Turn", button_style="primary")
btn_new = widgets.Button(description="New Session", button_style="info")
btn_restart = widgets.Button(description="Restart Runner", button_style="warning")
btn_save = widgets.Button(description="Save to Memory Bank", button_style="success")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output(layout=widgets.Layout(border="1px solid #cbd5e1", padding="8px", margin="6px 0"))


async def _do_action(kind: str) -> None:
    with out:
        if kind == "send" and prompt_box.value.strip():
            if win_slider.value != WORKBENCH["window"]:
                WORKBENCH["window"] = win_slider.value
                WORKBENCH["runner"] = _window_runner(WORKBENCH["window"])
            await chat(WORKBENCH["runner"], WORKBENCH["session"], prompt_box.value.strip())
        elif kind == "new":
            WORKBENCH["counter"] += 1
            WORKBENCH["session"] = await create_lab_session(f"playground_{WORKBENCH['counter']}")
        elif kind == "restart":
            WORKBENCH["runner"] = _window_runner(WORKBENCH["window"])
            print("\nRunner restarted. The session reloads from SessionService on the next turn.")
        elif kind == "save":
            await save_session_to_memory_bank(WORKBENCH["session"])


def _launch(kind: str) -> None:
    task = asyncio.ensure_future(_do_action(kind))

    def _report(t: asyncio.Task) -> None:
        if not t.cancelled() and t.exception() is not None:
            out.append_stdout(f"\nError during '{kind}': {t.exception()!r}\n")

    task.add_done_callback(_report)


btn_send.on_click(lambda _: _launch("send"))
btn_new.on_click(lambda _: _launch("new"))
btn_restart.on_click(lambda _: _launch("restart"))
btn_save.on_click(lambda _: _launch("save"))
btn_clear.on_click(lambda _: out.clear_output())

display(widgets.VBox([prompt_box, widgets.HBox([btn_send, btn_new, btn_restart, btn_save, btn_clear, win_slider]), out]))

---

## Recap: Check Your Understanding

Answer each question before opening the answers.

1. A tool writes `temp:daily_cost`. Where can you find that value after the turn ends?
2. The server restarts in the middle of INC-4092. Which object kept the ticket state, and why did a brand-new `Runner` still know it?
3. In `session_delta`, the agent knew your current language **and** the INC-4092 history. Which mechanism supplied each fact?

<details>
<summary>Answers</summary>

1. Nowhere. `temp:*` keys exist only for the rest of the turn and are stripped before the event is saved, so they never reach `SessionService` or the Console.
2. `SessionService`. The `Runner` holds no conversation data; on each turn it reloads the session (state and events) from `SessionService`.
3. The current language came from `user:preferred_language`, user-scoped state stored by `SessionService` and injected into the instruction. The INC-4092 history came from Memory Bank, saved with `add_events_to_memory()` and recalled by `preload_memory_tool`.

</details>

---

## Cleanup and Reset

* `DELETE_LAB_SESSIONS`: deletes every session this notebook created.
* `DELETE_AGENT_ENGINE`: deletes the engine and all of its sessions and memories. Do this when you finish.
* Local mode: re-run Step 2B to start fresh.

In [ ]:
# @title Cleanup: Delete Lab Sessions and (Optionally) the Agent Engine
DELETE_LAB_SESSIONS = True  # @param {type:"boolean"}
DELETE_AGENT_ENGINE = False  # @param {type:"boolean"}

if DELETE_LAB_SESSIONS:
    for sid, label in list(SESSION_LABELS.items()):
        try:
            await session_service.delete_session(app_name=APP_NAME, user_id=USER_ID, session_id=sid)
            print(f"Deleted session `{label}` ({sid})")
        except Exception as exc:
            print(f"Could not delete `{label}` ({sid}): {exc!r}")
        SESSION_LABELS.pop(sid, None)
    XRAY_HISTORY.clear()

if DELETE_AGENT_ENGINE and USING_CLOUD_AGENT_ENGINE:
    engine_name = f"projects/{ACTIVE_PROJECT}/locations/{ACTIVE_LOCATION}/reasoningEngines/{ACTIVE_ENGINE_ID}"
    vertex_client().agent_engines.delete(name=engine_name, force=True)
    os.environ.pop("MEMORY_LAB_ENGINE", None)
    print(f"Deleted Agent Engine {ACTIVE_ENGINE_ID}. Re-run Step 2B to create a new one.")